In [1]:
# Campaign 3: Emotions
emotions_labels = {
    "E01": "focus",
    "E02": "distraction",
    "E03": "stress",
    "E04": "relaxation",
    "E05": "depression",
    "E06": "excitement"
}

In [90]:
#Radar
import pickle
import numpy as np
import os

class RadarSensor:
    """A container for a single radar unit's data and configuration."""

    def __init__(self, translation, transform_func):
        self.translation = translation
        self.transform_func = transform_func  # Function to handle coordinate mapping

    def _load_data(self, path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
            # Filter empty or invalid frames (logic from ground script)
            cleaned = [arr for arr in data if isinstance(arr, np.ndarray) and arr.size > 0]
            # Fallback if cleaning removed everything or list was raw (logic from ceiling script)
            return np.concatenate(cleaned, axis=0) if cleaned else data
        
        
    def load_and_transform_data(self, path):
        """Loads data from a file and applies the transformation."""
        frames = self._load_data(path)
        return self.transform_func(frames, self.translation)
        
    
        
# --- Specific Math Logic for Ceiling ---
def ceiling_transform(points, translation):
    time_stamp=points[:, 0:1]  # First column is timestamp
    # Logic from original script: z = -points[:,1], y = points[:,2], x = points[:,3]
    z = -points[:, 1]
    y = points[:, 2]
    x = points[:, 3]
    translated = np.stack([x, y, z], axis=1) + translation
    return np.hstack([time_stamp, translated])

# --- Specific Math Logic for Ground ---
def ground_transform_factory(deg):
    """Returns a transformation function locked to a specific angle."""
    theta_rad = np.deg2rad(deg)
    # Pre-calculate R matrix
    R = np.array([
        [np.cos(theta_rad), -np.sin(theta_rad), 0],
        [np.sin(theta_rad), np.cos(theta_rad), 0],
        [0, 0, 1]
    ])

    def transform(points, translation):
        time_stamp=points[:, 0:1]  # First column is timestamp
        # Apply rotation then translation
        # Note: points[:, 1] is X, points[:, 2] is Y, points[:, 3] is Z
        res = R @ [points[:, 1], points[:, 2], points[:, 3]]
        x, y, z = res[0], res[1], res[2]
        translated = np.stack([x, y, z], axis=1) + translation
        return np.hstack([time_stamp, translated])

    return transform

In [91]:
sensors_info=[]

#Ceiling
FILES = ['0.pkl', '1.pkl', '2.pkl', '3.pkl', '4.pkl']
TRANSLATIONS = [
    np.array([-2, 4, 5]), np.array([2, 4, 5]), np.array([0, 0, 5]),
    np.array([-2, -4, 5]), np.array([2, -4, 5])
]
sensors_info.extend(list(zip(FILES, TRANSLATIONS,[ceiling_transform for _ in FILES])))

#Ground
FILES = ['5.pkl', '6.pkl', '7.pkl', '8.pkl', '9.pkl', '10.pkl', '11.pkl', '12.pkl']
THETA_DEGS = [180, 135, 90, 45, 0, -45, -90, -135]
TRANSLATIONS = [
    np.array([1.5, 0, 1.3]), np.array([1.06, -1.06, 1.3]),
    np.array([0, -1.5, 1.3]), np.array([-1.06, -1.06, 1.3]),
    np.array([-1.5, 0, 1.3]), np.array([-1.06, 1.06, 1.3]),
    np.array([0, 1.5, 1.3]), np.array([1.06, 1.06, 1.3])
]
sensors_info.extend(list(zip(FILES, TRANSLATIONS,[ground_transform_factory(degree) for degree in THETA_DEGS])))

In [92]:
sensors = []
for file_name, translation, transform_func in sensors_info:
    s = RadarSensor(
        translation=translation,
        transform_func=transform_func
    )
    sensors.append(s)

In [94]:
import glob

frame_data={}
users = glob.glob(f"E:/CoDaS Project/archive/Radar/C3/*")
for user in users:
    user_id = os.path.basename(user)
    frame_data[user_id] = {}
    for class_key in emotions_labels.keys():
        path = f"{user}/{class_key}/*/*.pkl"
        pkl_files = glob.glob(path)
        
        frame_data[user_id][class_key] = {}
        for file in pkl_files:
            sensor_id=os.path.basename(file).replace('.pkl', '')
            sensor_id=int(sensor_id)
            frames = sensors[sensor_id].load_and_transform_data(file)
            frame_data[user_id][class_key][sensor_id] = frames

In [105]:
for sensor in frame_data['U44']['E06'].keys():
    print(f"Data shape for sensor {sensor}: {frame_data['U44']['E06'][sensor].shape}")

Data shape for sensor 0: (31, 4)
Data shape for sensor 1: (269, 4)
Data shape for sensor 10: (408, 4)
Data shape for sensor 11: (298, 4)
Data shape for sensor 12: (263, 4)
Data shape for sensor 2: (377, 4)
Data shape for sensor 3: (9, 4)
Data shape for sensor 4: (172, 4)
Data shape for sensor 5: (595, 4)
Data shape for sensor 6: (215, 4)
Data shape for sensor 7: (475, 4)
Data shape for sensor 8: (120, 4)
Data shape for sensor 9: (303, 4)
